In [0]:
from pyspark.sql.types import *
cust_schema= StructType([StructField('Name', StringType(), True), StructField('City', StringType(), True), StructField('Age', IntegerType(), True), StructField('proof', StringType(), True)])
df=spark.read.format("csv").option("header","true").schema(cust_schema).load("/Volumes/databricks_practice/inputdb/customerdata/customers_v2.csv")
df.display()

In [0]:
##creating delta table
df.write.saveAsTable("databricks_practice.outputdb.tbl_customer_dlta_v2")

In [0]:
%sql
create table databricks_practice.outputdb.tbl_customer_dlta2_v2
USING DELTA
AS 
SELECT * FROM databricks_practice.outputdb.tbl_customer_dlta_v2;




In [0]:
%sql
MERGE INTO databricks_practice.outputdb.tbl_customer_dlta AS tar
USING databricks_practice.outputdb.tbl_customer_dlta2_v2 AS src
ON tar.Name = src.Name 
WHEN MATCHED THEN 
UPDATE databricks_practice.outputdb.tbl_customer_dlta 
SET City =src.City
   ,Age =src.Age
   ,proof =src.proof
WHEN NOT MATCHED THEN 
INSERT  databricks_practice.outputdb.tbl_customer_dlta (Name,City,Age,proof)
VALUES (src.Name,src.City,src.Age,src.proof)

In [0]:
from delta.tables import DeltaTable
# Load target Delta table
target_tbl = DeltaTable.forName(spark, "databricks_practice.outputdb.tbl_customer_dlta")
# Load or create your updates DataFrame
updatesDF = spark.table("databricks_practice.inputdb.tbl_customer_dlta2_v2")

target_tbl.alias("target").merge(updatesDF.alias("source"),
                                 "target.customer_id = source.customer_id"
                                 ).whenMatchedUpdate(set={"proof": "source.proof",
    "City": "source.City",
    "Age": "source.Age"}
                                                     ).whenNotMatchedInsert(values={ "customer_id": "source.customer_id",
    "Name": "source.Name",
    "proof": "source.proof",
    "City": "source.City",
    "Age": "source.Age"}).execute()